# The Rete Algorithm

A refresher on the pattern-matching engine that powers production rule systems
(CLIPS, Jess, Drools, Soar).

**Domain:** Symbolic AI & Logic  ·  **recommended addition**  ·  **runnable:** yes


## 1. What & Why

The **Rete algorithm** (Charles Forgy, 1979 — *rete* is Latin for "net") is how
production-rule engines match a large set of **rules** against a large set of
**facts** efficiently.

The problem it solves: a rule's left-hand side (LHS) is a conjunction of patterns,
e.g. *"a Customer who is `gold` **and** has an Order over $100"*. A naive engine
re-checks every rule against every combination of facts on every cycle. With `R`
rules, each joining `k` conditions over `F` facts, that is on the order of
`R · F^k` tests **per cycle** — and a rule firing typically changes only one or
two facts, so almost all of that work is recomputed identically each time.

Rete's insight: **don't recompute, remember.** It compiles the rules once into a
dataflow network and stores the *partial matches* at every node. When a fact is
added or removed, only the **delta** flows through the network and updates the
cached state. Matching cost becomes roughly proportional to the *change*, not to
the total size of working memory. This is the classic **time–space trade-off**:
spend memory on partial-match state to avoid re-deriving it.

**Reach for it when:** you have many rules, a slowly-changing fact base, and you
run the match/act loop repeatedly (expert systems, business-rule engines,
event/CEP systems, some game AI).

**Skip it when:** you have few rules or facts (a plain loop is simpler and uses
no memory), facts churn so fast that cached state is constantly invalidated, or
your matching is mostly a single lookup rather than multi-condition joins. The
stored state is also memory-hungry — pathological rule sets can blow up.


## 2. Mental Model

Think of Rete as a **compiled, incrementally-maintained query plan** — like a
database engine that keeps a materialized view up to date as rows change, instead
of re-running the `SELECT` from scratch each time.

```
facts ──▶  ALPHA network          BETA network            ▶ agenda (conflict set)
           (per-condition         (joins across
            single-fact tests)     conditions)

  (Customer ?c gold) ─┐
                      ├─▶ [join on ?c] ─▶ [join on ...] ─▶ rule activation
  (Order ?c ?amt) ────┘        │
                               └─ caches partial matches (tokens)
```

- The **alpha network** filters single facts by the intra-condition tests
  (right type, constant fields). Each surviving fact lands in an **alpha memory**.
- The **beta network** is a chain of **join nodes** that combine facts across
  conditions wherever they share a variable (`?c` above). Each join node caches
  its partial matches as **tokens** in a **beta memory**.
- A fact added to working memory enters as a token and propagates *only as far as
  it matches*. A fact retracted removes exactly the tokens that depended on it.

The whole thing runs the **match → resolve → act** cycle: Rete *matches* (keeps
the agenda of fully-satisfied rules current), a conflict-resolution strategy
*resolves* which activation to run, and its right-hand side *acts* by asserting /
retracting facts — which feeds the next delta back into the net.


## 3. Key Concepts

- **Working memory & WMEs** — the fact base; each fact is a *working-memory
  element*. Asserting/retracting facts is what drives the engine.
- **Production rule** — `LHS ⇒ RHS`. The LHS is a conjunction of **conditions**
  (patterns with variables); the RHS is the action run when the LHS matches.
- **Alpha network / alpha memory** — nodes performing *intra-element* tests on a
  single fact (relation, constant fields). Matching facts are stored per condition.
- **Beta network / join nodes / beta memory** — nodes performing *inter-element*
  tests: they **join** an upstream partial match with an alpha memory, enforcing
  that **shared variables** are consistent. Cached partial matches are **tokens**.
- **Token** — a partial match: an ordered set of facts satisfying the first *i*
  conditions of a rule, plus the variable bindings they imply.
- **Node sharing** — identical conditions/sub-networks across rules are compiled
  **once** and shared, so common tests are evaluated a single time.
- **Conflict set / agenda** — the set of rule activations whose LHS is fully
  satisfied. **Conflict resolution** (recency, specificity, salience/priority,
  refraction) picks which one fires.
- **Truth maintenance / retraction** — because matches are cached against the
  facts that produced them, retracting a fact incrementally removes exactly the
  activations it supported. No re-derivation needed.
- **Incrementality** — the headline property: cost tracks the *change* to working
  memory, not its total size.


## 4. Setup

Nothing to install — the worked examples build a small but faithful Rete network
in **pure Python** so you can watch tokens propagate and count the work done.

The last cell *optionally* shows the same rule in a real CLIPS-style engine
(`experta`) if it happens to be importable; it skips cleanly otherwise, so the
notebook always runs top-to-bottom in a fresh kernel.


In [ ]:
import sys
print("Python", sys.version.split()[0])
print("No third-party packages required for the core examples.")


## 5. Worked Examples

We implement a compact Rete in pure Python. Facts are tuples like
`("parent", "ann", "bob")`; conditions are the same shape with variables written
as strings starting with `?`. The engine maintains **alpha memories** (facts per
condition) and **beta memories** (tokens per partial match), and propagates each
asserted/retracted fact incrementally — exactly the Rete idea.


In [ ]:
from collections import defaultdict


def match_const(pattern, fact):
    """Alpha test: does one fact match a condition's relation + constant fields?

    Returns the variable bindings it implies, or None on mismatch.
    """
    if not (isinstance(fact, tuple) and pattern[0] == fact[0]
            and len(pattern) == len(fact)):
        return None
    binds = {}
    for p, f in zip(pattern[1:], fact[1:]):
        if isinstance(p, str) and p.startswith("?"):
            if binds.get(p, f) != f:      # repeated var within one condition
                return None
            binds[p] = f
        elif p != f:                       # constant field must match exactly
            return None
    return binds


def unify(left, right):
    """Beta/join test: merge two binding dicts; None if a shared var conflicts."""
    merged = dict(left)
    for k, v in right.items():
        if merged.setdefault(k, v) != v:
            return None
    return merged


class Rete:
    """A minimal but faithful Rete: alpha memories + a beta join chain per rule."""

    def __init__(self, rules):
        # rules: list of (name, [condition, ...])
        self.rules = rules
        self.alpha = defaultdict(list)   # (rule, cond_index) -> [(fact, binds)]
        self.beta = defaultdict(list)    # (rule, level)       -> [(facts, binds)]
        for r, _ in enumerate(rules):
            self.beta[(r, 0)] = [((), {})]   # dummy top node: one empty token
        self.joins = 0                   # instrumentation: join attempts done

    def _propagate(self, r, level, facts, binds):
        """A new token entered beta[(r, level)]; push it down the join chain."""
        self.beta[(r, level)].append((facts, binds))
        conds = self.rules[r][1]
        if level == len(conds):
            return                       # reached the end -> a full activation
        for fact, fb in self.alpha[(r, level)]:   # join against known facts
            self.joins += 1
            m = unify(binds, fb)
            if m is not None:
                self._propagate(r, level + 1, facts + (fact,), m)

    def assert_fact(self, fact):
        for r, (_, conds) in enumerate(self.rules):
            for ci, cond in enumerate(conds):
                fb = match_const(cond, fact)
                if fb is None:
                    continue
                self.alpha[(r, ci)].append((fact, fb))
                # right-activate: join the new fact with existing left tokens
                for facts, binds in list(self.beta[(r, ci)]):
                    self.joins += 1
                    m = unify(binds, fb)
                    if m is not None:
                        self._propagate(r, ci + 1, facts + (fact,), m)

    def retract(self, fact):
        # A token is valid iff every fact it contains is still present, so we
        # simply drop every memory entry that mentions the retracted fact.
        for key, mem in self.alpha.items():
            self.alpha[key] = [(f, b) for f, b in mem if f != fact]
        for (r, level), mem in list(self.beta.items()):
            if level == 0:
                continue                 # keep the dummy empty token
            self.beta[(r, level)] = [(fs, b) for fs, b in mem if fact not in fs]

    def matches(self):
        """Fully-satisfied activations: (rule_name, bindings) for each rule."""
        out = []
        for r, (name, conds) in enumerate(self.rules):
            for facts, binds in self.beta[(r, len(conds))]:
                out.append((name, {k: binds[k] for k in sorted(binds)}))
        return out


print("Rete engine defined.")


### Example 1 — joins, incrementality, and the win over naive matching

A `grandparent` rule joins the `parent` relation with itself on the shared
middle person `?y`:

```
(parent ?x ?y) ∧ (parent ?y ?z)  ⇒  grandparent(?x, ?z)
```

We load a family tree, then add **one** more fact and watch how little extra work
Rete does — and compare that against a naive matcher that re-checks every pair.


In [ ]:
rules = [("grandparent", [("parent", "?x", "?y"), ("parent", "?y", "?z")])]

facts = [
    ("parent", "ann", "bob"),
    ("parent", "bob", "carol"),
    ("parent", "carol", "dave"),
    ("parent", "ann", "beth"),
    ("parent", "beth", "carl"),
]

net = Rete(rules)
for f in facts:
    net.assert_fact(f)

print("grandparent matches after loading", len(facts), "facts:")
for name, b in sorted(net.matches(), key=lambda m: (m[1]["?x"], m[1]["?z"])):
    print(f"  {b['?x']} -> {b['?z']}")
print("join attempts so far:", net.joins)

# Now assert ONE more edge and measure the incremental work.
before = net.joins
net.assert_fact(("parent", "dave", "erin"))
print("\njoin attempts for the single new fact:", net.joins - before)
pairs = {(b["?x"], b["?z"]) for _, b in net.matches()}
print("new grandparent matches now include carol -> erin:", ("carol", "erin") in pairs)


In [ ]:
# Naive baseline: recompute ALL grandparent matches from scratch, counting work.
def naive_grandparents(facts):
    work = 0
    out = []
    parents = [f for f in facts if f[0] == "parent"]
    for _, x, y in parents:
        for _, y2, z in parents:
            work += 1                     # every pair is re-tested
            if y == y2:
                out.append((x, z))
    return out, work

all_facts = facts + [("parent", "dave", "erin")]
out, work = naive_grandparents(all_facts)
print("naive matches:", sorted(out))
print("naive pair-tests to recompute from scratch:", work)
print("Rete join attempts for the incremental add:", net.joins - before)
print("\nThe naive matcher redoes O(F^2) work every cycle; Rete touched only the"
      "\nfew tokens the new fact actually reached.")


### Example 2 — alpha (constant) tests, retraction & truth maintenance

Two conditions, one with a **constant** field, joined on the person `?p`:

```
(member ?p club) ∧ (active ?p yes)  ⇒  eligible(?p)
```

The constant `club`/`yes` fields are filtered in the **alpha** network. Then we
**retract** a fact and confirm the dependent activation disappears automatically —
Rete's built-in truth maintenance, with no re-derivation.


In [ ]:
rules2 = [("eligible", [("member", "?p", "club"), ("active", "?p", "yes")])]
net2 = Rete(rules2)

for f in [
    ("member", "alice", "club"),
    ("member", "bob", "club"),
    ("active", "alice", "yes"),
    ("active", "bob", "no"),     # bob is inactive -> filtered out at the alpha node
]:
    net2.assert_fact(f)

print("eligible after asserts:", [b["?p"] for _, b in net2.matches()])

# Truth maintenance: retract alice's active status; her activation must vanish.
net2.retract(("active", "alice", "yes"))
print("eligible after retracting alice's active=yes:",
      [b["?p"] for _, b in net2.matches()])

# Re-assert it and the match returns, incrementally.
net2.assert_fact(("active", "alice", "yes"))
print("eligible after re-asserting:", [b["?p"] for _, b in net2.matches()])


In [ ]:
# Optional: the same rule in a real CLIPS-style engine (experta), if available.
# experta isn't an API/large-download dependency, so we just probe the import and
# skip cleanly when it's absent -- the notebook still runs end to end.
try:
    from experta import KnowledgeEngine, Rule, Fact, Field   # noqa: F401
    HAVE_EXPERTA = True
except Exception:
    HAVE_EXPERTA = False

if HAVE_EXPERTA:
    from experta import KnowledgeEngine, Rule, Fact

    class Member(Fact):
        pass

    class Active(Fact):
        pass

    class Eligibility(KnowledgeEngine):
        @Rule(Member(name="p", group="club"), Active(name="p", state="yes"))
        def eligible(self, p):
            print("experta: eligible ->", p)

    ke = Eligibility()
    ke.reset()
    ke.declare(Member(name="alice", group="club"))
    ke.declare(Active(name="alice", state="yes"))
    ke.run()
else:
    print("experta not installed -- showing the rule shape only.")
    print("Install with:  %pip install experta")
    print()
    print("    @Rule(Member(name='p', group='club'), Active(name='p', state='yes'))")
    print("    def eligible(self, p):")
    print("        print('eligible ->', p)")
    print()
    print("The pure-Python Rete above already demonstrates the same matching.")


## 6. Gotchas & Pitfalls

- **Memory blow-up.** Beta memories store every partial match. A rule whose early
  conditions are unselective (match many facts) before a later, restrictive join
  can cache a combinatorial number of tokens. **Order conditions most-restrictive
  first** so few tokens survive into the beta chain.
- **Join order matters, a lot.** The engine joins conditions left-to-right. A poor
  order produces huge intermediate token sets even when the final result is tiny.
  This is the rule-engine analogue of SQL join-order optimization.
- **Cross-products from unshared variables.** If two conditions share *no*
  variable, their join is a full Cartesian product. Usually a sign of a missing
  join key in the rule.
- **Cost is in `assert`/`retract`, not in `run`.** Newcomers profile the wrong
  phase. The match work happens as facts change; the agenda is then cheap to read.
- **Refraction & loops.** Without refraction (not re-firing the same activation on
  the same facts) a rule whose RHS asserts a fact its LHS matches will loop
  forever. Real engines track this; a hand-rolled Rete must too.
- **Modify = retract + re-assert.** "Updating" a fact invalidates every token
  derived from the old value and rebuilds from the new one — not free. Modifying a
  field that feeds an unselective condition can be surprisingly expensive.
- **Classic Rete keeps stale alpha state across runs.** The original algorithm
  doesn't reclaim memory aggressively; long-lived engines need care (Rete-II /
  Rete-NT and engines like Drools added optimizations such as right/left unlinking
  to avoid maintaining nodes that currently can't match).


## 7. When to Use vs Alternatives

| Approach | Best when | Trade-off |
|---|---|---|
| **Rete** (CLIPS, Jess, Drools) | Many rules, many facts, **incremental** changes, repeated match cycles | High memory for cached partial matches; complex to implement |
| **TREAT** (Miranker) | Frequent **retractions**; memory-tight | Recomputes some joins Rete would cache — cheaper memory, more match work |
| **LEAPS** (lazy) | Huge working memory, few rules fire | Lazy/iterator-based — avoids materializing all matches, harder to reason about |
| **Plain nested loops / `if` cascade** | A handful of rules or facts | No setup, no memory state; O(F^k) per cycle, but k and F are tiny |
| **Relational / Datalog query** | Matching *is* a query over relations, set-at-a-time | Great for bulk evaluation; weaker at incremental assert/retract loops |
| **Decision tree / rule list (ML)** | Rules learned from data, single-fact classification | No joins across facts, no working-memory dynamics |

Rules of thumb:

- **Few rules or a one-shot match?** Skip Rete; loop or query directly.
- **Long-running engine with a churning fact base and multi-condition rules?**
  That's exactly Rete's sweet spot — and you should reach for a mature engine
  (Drools, CLIPS, Jess) rather than rolling your own.
- **Retraction-heavy and memory-constrained?** TREAT-style matching trades cached
  state for recomputation and can win.
- See also the [`clips`](./clips.ipynb) notebook for a production engine that uses
  Rete under the hood, and [`datalog`](./datalog.ipynb) for the set-at-a-time,
  query-oriented alternative.


## 8. Resources

- **Forgy, C. (1982). "Rete: A Fast Algorithm for the Many Pattern/Many Object
  Pattern Match Problem."** *Artificial Intelligence* 19(1) — the original paper.
  https://www.csl.sri.com/users/mwfong/Technical/RETE%20Match%20Algorithm%20-%20Forgy%20OCR.pdf
- **Doorenbos, R. (1995). "Production Matching for Large Learning Systems"**
  (CMU PhD thesis) — the clearest modern, implementation-level walkthrough of
  Rete and Rete/UL. https://reports-archive.adm.cs.cmu.edu/anon/1995/CMU-CS-95-113.pdf
- **CLIPS** — the canonical open-source Rete engine and its reference docs.
  https://www.clipsrules.net/
- **Drools documentation** — a widely-used modern business-rules engine
  (Rete-OO / PHREAK). https://www.drools.org/
- **Wikipedia: Rete algorithm** — a solid, well-illustrated overview.
  https://en.wikipedia.org/wiki/Rete_algorithm


YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def token_counts(conditions, facts):
    """How many partial matches survive after each condition is joined in."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE